# Self-Steering Alpha + Layer Sweep

The pilot study (layer=16, alpha=4.0) showed weak or absent self-steering for some traits. This notebook sweeps over layers and alpha values to find the regime where self-steering actually works, before drawing conclusions about cross-trait transfer.

**Sweep grid:**
- Layers: every 4th layer (0, 4, 8, ..., 32) + layer 16 from pilot
- Alphas: 1, 2, 4, 8, 16

**For each (layer, alpha):** steer trait X → measure trait X (diagonal only). Judge via logprob scoring as in the main pipeline.

## Setup

In [ ]:
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/junekhunter/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/steering_independence
    !pip install -q transformers torch openai seaborn pyyaml tqdm scipy bitsandbytes
    !pip install -q -e ../../niels/propensities
    from google.colab import drive
    drive.mount('/content/drive')
    _drive = '/content/drive/MyDrive/spar-ood-propensities/june/steering_independence/outputs'
    os.makedirs(_drive, exist_ok=True)
    !ln -sfn {_drive} outputs
else:
    %cd {os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(__file__)}

In [ ]:
import sys, json, asyncio, math
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    pass

propensities_root = str(Path('.').resolve().parent.parent / 'niels' / 'propensities')
if propensities_root not in sys.path:
    sys.path.insert(0, propensities_root)

import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
from tqdm.auto import tqdm
from openai import AsyncOpenAI

from trait_registry import ALL_TRAITS, LABELS, get_trait_spec, load_test_questions
from utils import load_model, get_model_layers, SteeringHook
from cache import JudgeCache

## Config

In [ ]:
with open('config.yaml') as f:
    config = yaml.safe_load(f)

# --- Sweep parameters (edit these) ---
LAYERS = sorted(set(list(range(0, 36, 4)) + [16]))  # [0, 4, 8, 12, 16, 20, 24, 28, 32]
ALPHAS = [1.0, 2.0, 4.0, 8.0, 16.0]
TRAITS = config.get('traits') or ALL_TRAITS
MAX_TEST_QUESTIONS = 20  # cap per trait to keep sweep tractable; set None for all

# Generation params from pilot
MAX_NEW_TOKENS = config.get('behavioral', {}).get('max_new_tokens', 512)
TEMPERATURE = config.get('behavioral', {}).get('temperature', 0.7)
BATCH_SIZE = config.get('behavioral', {}).get('batch_size', 16)
MODEL_ID = config['model_id']
LOAD_IN_4BIT = config.get('load_in_4bit', False)

# Judge
JUDGE_MODEL = config.get('judge', {}).get('model', 'openai/gpt-4o-mini')
JUDGE_CONCURRENCY = config.get('judge', {}).get('concurrency', 20)

# Output
SWEEP_DIR = Path(config['output_dir']) / 'self_steering_sweep'
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_ID}")
print(f"Layers: {LAYERS}")
print(f"Alphas: {ALPHAS}")
print(f"Traits: {len(TRAITS)}")
print(f"Max test questions per trait: {MAX_TEST_QUESTIONS}")
print(f"Total grid points: {len(LAYERS) * len(ALPHAS) * len(TRAITS)}")

## Load model + test questions

In [ ]:
model, tokenizer = load_model(MODEL_ID, load_in_4bit=LOAD_IN_4BIT)
n_layers = len(get_model_layers(model))
print(f"Model loaded: {n_layers} layers")

# Validate sweep layers
assert all(l < n_layers for l in LAYERS), f"Some layers exceed model depth ({n_layers})"

# Load test questions
test_qs = {}
for trait in TRAITS:
    qs = load_test_questions(trait)
    if MAX_TEST_QUESTIONS:
        qs = qs[:MAX_TEST_QUESTIONS]
    test_qs[trait] = qs
    print(f"  {trait}: {len(qs)} test questions")

## Generate baseline (no steering)

In [ ]:
from behavioral_steering import _generate_responses, _save_jsonl, _load_jsonl

gen_dir = SWEEP_DIR / 'generations'
gen_dir.mkdir(parents=True, exist_ok=True)

# Baseline: one set per trait (shared across all layer/alpha combos)
for trait in tqdm(TRAITS, desc="Baseline"):
    out_path = gen_dir / f"baseline_{trait}.jsonl"
    if out_path.exists():
        print(f"  {trait}: cached ({len(_load_jsonl(out_path))} responses)")
        continue
    results = _generate_responses(model, tokenizer, test_qs[trait], MAX_NEW_TOKENS, TEMPERATURE, BATCH_SIZE)
    _save_jsonl(results, out_path)
    print(f"  {trait}: generated {len(results)} responses")

## Steered generation sweep

For each trait, iterate over (layer, alpha). Vectors were already extracted for all layers by `extract_vectors.py`.

In [ ]:
vec_dir = Path(config['output_dir']) / 'vectors'

total = len(TRAITS) * len(LAYERS) * len(ALPHAS)
pbar = tqdm(total=total, desc="Steered sweep")

for trait in TRAITS:
    for layer in LAYERS:
        vec_path = vec_dir / f"{trait}_layer{layer}.pt"
        if not vec_path.exists():
            print(f"  WARNING: {vec_path} not found, skipping")
            pbar.update(len(ALPHAS))
            continue
        steering_vec = torch.load(vec_path, weights_only=True)

        for alpha in ALPHAS:
            out_path = gen_dir / f"{trait}_L{layer}_a{alpha}.jsonl"
            if out_path.exists():
                pbar.update(1)
                continue

            with SteeringHook(model, layer, steering_vec, alpha=alpha):
                results = _generate_responses(
                    model, tokenizer, test_qs[trait],
                    MAX_NEW_TOKENS, TEMPERATURE, BATCH_SIZE,
                )
            _save_jsonl(results, out_path)
            pbar.update(1)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pbar.close()
print("Generation complete.")

In [ ]:
# Free GPU memory before judge calls
del model, tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
import gc; gc.collect()
print("Model freed.")

## Judge scoring

In [ ]:
from behavioral_steering import _judge_single, _aggregate_0_100_score

api_key = os.environ.get('OPENROUTER_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        pass
assert api_key, "OPENROUTER_API_KEY not found"

client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
sem = asyncio.Semaphore(JUDGE_CONCURRENCY)
cache = JudgeCache(str(SWEEP_DIR / 'judge_cache.db'))


async def judge_file(trait: str, gen_path: Path, source_label: str, layer: int, alpha: float) -> list[float]:
    """Judge all responses in a generation file. Returns list of valid scores."""
    spec = get_trait_spec(trait)
    records = _load_jsonl(gen_path)

    async def _judge_one(rec):
        cache_key = JudgeCache.make_key(
            model_id=MODEL_ID,
            question_id=rec['id'],
            source_trait=source_label,
            target_trait=trait,
            alpha=alpha,
            layer=layer,
            metric=spec.primary_metric,
        )
        cached = cache.get(cache_key)
        if cached is not None:
            return cached
        score = await _judge_single(client, spec.judge_prompt, rec['question'], rec['response'], sem, judge_model=JUDGE_MODEL)
        if score is not None:
            cache.put(cache_key, score)
        return score

    scores = await asyncio.gather(*[_judge_one(r) for r in records])
    return [s for s in scores if s is not None]


print(f"Judge ready: {JUDGE_MODEL}, concurrency={JUDGE_CONCURRENCY}")

In [ ]:
# Judge baseline + all steered files, collect into a results table
rows = []

# Baseline scores
for trait in tqdm(TRAITS, desc="Judge baseline"):
    bl_path = gen_dir / f"baseline_{trait}.jsonl"
    scores = await judge_file(trait, bl_path, 'baseline', layer=0, alpha=0.0)
    rows.append({'trait': trait, 'layer': None, 'alpha': 0.0, 'condition': 'baseline',
                 'mean_score': np.mean(scores), 'std_score': np.std(scores), 'n': len(scores)})

# Steered scores
for trait in tqdm(TRAITS, desc="Judge steered"):
    bl_scores = [r for r in rows if r['trait'] == trait and r['condition'] == 'baseline']
    bl_mean = bl_scores[0]['mean_score'] if bl_scores else 0

    for layer in LAYERS:
        for alpha in ALPHAS:
            gen_path = gen_dir / f"{trait}_L{layer}_a{alpha}.jsonl"
            if not gen_path.exists():
                continue
            scores = await judge_file(trait, gen_path, trait, layer, alpha)
            if not scores:
                continue
            rows.append({
                'trait': trait, 'layer': layer, 'alpha': alpha,
                'condition': 'steered',
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
                'n': len(scores),
            })

results_df = pd.DataFrame(rows)
results_df.to_csv(SWEEP_DIR / 'sweep_results.csv', index=False)
print(f"Judged {len(results_df)} conditions. Saved to {SWEEP_DIR / 'sweep_results.csv'}")
results_df.head(10)

## Compute deltas (steered - baseline)

In [ ]:
# Build baseline lookup
baseline = results_df[results_df['condition'] == 'baseline'].set_index('trait')['mean_score'].to_dict()

# Compute delta for steered rows
steered = results_df[results_df['condition'] == 'steered'].copy()
steered['delta'] = steered.apply(lambda r: r['mean_score'] - baseline.get(r['trait'], 0), axis=1)
steered['label'] = steered['trait'].map(LABELS)

steered.to_csv(SWEEP_DIR / 'sweep_deltas.csv', index=False)
print("Baseline scores:")
for trait, score in baseline.items():
    print(f"  {LABELS[trait]:25s} {score:.1f}")
print()
steered.sort_values('delta', ascending=False).head(10)

## Plot: Per-trait heatmaps (layer x alpha)

Each heatmap shows the score delta (steered - baseline) for one trait across all (layer, alpha) combinations.

In [ ]:
plot_dir = SWEEP_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

ncols = 3
nrows = math.ceil(len(TRAITS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

# Global color range for consistent comparison
vmax = max(abs(steered['delta'].min()), abs(steered['delta'].max()))
vmin = -vmax

for idx, trait in enumerate(TRAITS):
    ax = axes[idx // ncols][idx % ncols]
    trait_data = steered[steered['trait'] == trait]

    # Pivot to heatmap: rows=layers, cols=alphas
    pivot = trait_data.pivot(index='layer', columns='alpha', values='delta')
    pivot = pivot.reindex(index=LAYERS, columns=ALPHAS)

    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                vmin=vmin, vmax=vmax, ax=ax, cbar_kws={'label': 'Score delta'})
    ax.set_title(LABELS[trait], fontsize=12, fontweight='bold')
    ax.set_xlabel('Alpha')
    ax.set_ylabel('Layer')

# Hide unused axes
for idx in range(len(TRAITS), nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(f'Self-Steering Sweep: Score Delta by Layer x Alpha\n{MODEL_ID}', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(plot_dir / 'per_trait_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot: Aggregated across traits

Average delta across all traits at each (layer, alpha) point — shows the overall sweet spot.

In [ ]:
# Mean delta across traits
agg = steered.groupby(['layer', 'alpha'])['delta'].mean().reset_index()
pivot_agg = agg.pivot(index='layer', columns='alpha', values='delta').reindex(index=LAYERS, columns=ALPHAS)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot_agg, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax,
            cbar_kws={'label': 'Mean score delta'})
ax.set_title(f'Mean Self-Steering Delta (all {len(TRAITS)} traits)\n{MODEL_ID}', fontweight='bold')
ax.set_xlabel('Alpha')
ax.set_ylabel('Layer')
fig.tight_layout()
fig.savefig(plot_dir / 'aggregate_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Best point
best = agg.loc[agg['delta'].idxmax()]
print(f"\nBest (layer, alpha): layer={int(best['layer'])}, alpha={best['alpha']}, mean delta={best['delta']:.1f}")

## Plot: Line plots (delta vs alpha, one line per layer)

Shows how each layer's effectiveness scales with alpha — useful for spotting saturation or inversion at high alpha.

In [ ]:
ncols = 3
nrows = math.ceil(len(TRAITS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

cmap = plt.cm.viridis
layer_colors = {l: cmap(i / max(len(LAYERS) - 1, 1)) for i, l in enumerate(LAYERS)}

for idx, trait in enumerate(TRAITS):
    ax = axes[idx // ncols][idx % ncols]
    trait_data = steered[steered['trait'] == trait]

    for layer in LAYERS:
        ld = trait_data[trait_data['layer'] == layer].sort_values('alpha')
        if ld.empty:
            continue
        ax.plot(ld['alpha'], ld['delta'], 'o-', color=layer_colors[layer],
                label=f'L{layer}', markersize=4)

    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(LABELS[trait], fontweight='bold')
    ax.set_xlabel('Alpha')
    ax.set_ylabel('Score delta')
    ax.set_xscale('log', base=2)

# Shared legend
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', title='Layer', fontsize=8, title_fontsize=9)

for idx in range(len(TRAITS), nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(f'Self-Steering: Delta vs Alpha by Layer\n{MODEL_ID}', fontsize=14, y=1.01)
fig.tight_layout()
fig.subplots_adjust(right=0.88)
fig.savefig(plot_dir / 'line_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary: Best (layer, alpha) per trait

In [ ]:
# Per-trait best settings
best_per_trait = steered.loc[steered.groupby('trait')['delta'].idxmax()][
    ['trait', 'label', 'layer', 'alpha', 'delta', 'mean_score']
].sort_values('delta', ascending=False).reset_index(drop=True)

print("Best (layer, alpha) per trait:")
print("=" * 70)
for _, row in best_per_trait.iterrows():
    bl = baseline[row['trait']]
    print(f"  {row['label']:25s}  L{int(row['layer']):2d}  a={row['alpha']:5.1f}  "
          f"delta={row['delta']:+6.1f}  (baseline={bl:.1f} -> steered={row['mean_score']:.1f})")

# Traits where NO setting produced a meaningful positive delta
weak = best_per_trait[best_per_trait['delta'] < 3.0]
if len(weak) > 0:
    print(f"\nTraits with weak self-steering (best delta < 3):")
    for _, row in weak.iterrows():
        print(f"  {row['label']:25s}  best delta={row['delta']:+.1f}")

best_per_trait